# TUGAS BESAR DASAR KECERDASAN BUATAN (DKA)
## SISTEM ESTIMASI HARGA MOBIL BEKAS MERCEDES-BENZ MENGGUNAKAN LOGIKA FUZZY

---

**KELOMPOK / ANGGOTA:**
1. **Sheva Ilham Ramadhan** - NIM: **103012400259**
2. **Dhimas Rafi Noer Wicaksono** - NIM: **103012400259**

**KELAS: IF-48-02**


### Langkah 1: Import Library & Persiapan Awal
Mengimpor pustaka (library) dasar Python yang diperlukan untuk pemrosesan data, manipulasi matriks, visualisasi grafik (termasuk `seaborn` untuk Heatmap korelasi), dan metrik evaluasi (`sklearn` untuk MAE, MSE, RMSE, serta Confusion Matrix).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

### Langkah 2: Exploratory Data Analysis (EDA) & Data Preprocessing
Membaca berkas dataset riil Mercedes-Benz dari `data/merc.csv`, kemudian melakukan tahap **Preprocessing** (pembersihan data hilang, data duplikat) dan analisis **Korelasi (Heatmap)** antar parameter fitur.

In [2]:
data_mobil = pd.read_csv("data/merc.csv")
data_mobil.head()

#### 2.1. Pra-pemrosesan Data (Preprocessing)
Sebelum memodelkan logika fuzzy, kita membersihkan dataset dari data kosong (*missing values*) dan baris duplikat agar model yang dievaluasi didasarkan pada data yang bersih.

In [3]:
print("=== MENGECEK DATA KOSONG (MISSING VALUES) ===")
print(data_mobil.isnull().sum())

print("\n=== MENGECEK DATA DUPLIKAT ===")
print("Jumlah data duplikat:", data_mobil.duplicated().sum())

# Membersihkan data duplikat jika ada
data_mobil = data_mobil.drop_duplicates()
print("Dimensi data setelah menghapus duplikat:", data_mobil.shape)

=== MENGECEK DATA KOSONG (MISSING VALUES) ===
model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
dtype: int64

=== MENGECEK DATA DUPLIKAT ===
Jumlah data duplikat: 259
Dimensi data setelah menghapus duplikat: (12860, 9)


In [4]:
# Menyaring kolom data numerik yang digunakan sebagai parameter input/output fuzzy
data_mobil = data_mobil[['year', 'mileage', 'tax', 'mpg', 'engineSize', 'price']]
data_mobil.head()

#### 2.2. Pembagian Data (Train-Test Split)
Membagi dataset menjadi **80% Data Latih (Train Set)** untuk analisis statistik (EDA) & pembentukan aturan fuzzy, dan **20% Data Uji (Test Set)** untuk mengevaluasi performa prediksi model fuzzy di bagian akhir.

In [5]:
# Melakukan split dataset (80% train, 20% test)
train_data, test_data = train_test_split(data_mobil, test_size=0.2, random_state=42)
print("Dimensi Dataset Utama :", data_mobil.shape)
print("Dimensi Data Latih (Train Set) :", train_data.shape)
print("Dimensi Data Uji (Test Set)  :", test_data.shape)

Dimensi Dataset Utama : (12860, 6)
Dimensi Data Latih (Train Set) : (10288, 6)
Dimensi Data Uji (Test Set)  : (2572, 6)


#### 2.3. Matriks Korelasi & Heatmap
Matriks korelasi digunakan untuk mengukur kekuatan hubungan linear antar variabel numerik. Nilainya berkisar antara -1 hingga 1. Visualisasi korelasi dilakukan menggunakan **Heatmap**.

In [6]:
# Menghitung matriks korelasi pada Train Set
corr_matrix = train_data.corr()
print("=== MATRIKS KORELASI (TRAIN SET) ===")
print(corr_matrix)

=== MATRIKS KORELASI (TRAIN SET) ===
                year   mileage       tax       mpg  engineSize     price
year        1.000000 -0.748036  0.009779 -0.092083   -0.144272  0.522830
mileage    -0.748036  1.000000 -0.148155  0.199771    0.058848 -0.531925
tax         0.009779 -0.148155  1.000000 -0.519747    0.348582  0.267973
mpg        -0.092083  0.199771 -0.519747  1.000000   -0.354590 -0.448607
engineSize -0.144272  0.058848  0.348582 -0.354590    1.000000  0.525541
price       0.522830 -0.531925  0.267973 -0.448607    0.525541  1.000000


In [7]:
# Menampilkan visualisasi Heatmap Korelasi pada Train Set
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap - Train Set (Mercedes-Benz)")
plt.show()

In [8]:
# Menampilkan histogram persebaran fitur pada Train Set
train_data.hist(figsize=(12,8))
plt.show()

### Langkah 3: Definisikan Fungsi Keanggotaan Fuzzy (Fuzzifikasi)
Fuzzifikasi adalah proses pemetaan nilai tegas (crisp value) ke dalam nilai linguistik fuzzy menggunakan fungsi keanggotaan. Di sini kita menggunakan **Fungsi Keanggotaan Segitiga** (Triangular Membership Function) karena sifatnya yang sederhana dan efisien.

In [9]:
def triangleMember(x, a, b, c):
    """Fungsi keanggotaan segitiga (triangular membership function).
    Parameter: x = nilai input, a = batas kiri, b = puncak, c = batas kanan.
    """
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x < c:
        return (c - x) / (c - b)
    return 0.0


In [10]:
# Pengujian fungsi kurva keanggotaan segitiga
triangleMember(20000, 0, 20000, 40000)

#### 3.1. Variabel Input: Mileage (Jarak Tempuh)
- Kategori: **Low** (0–40.000), **Medium** (30.000–70.000), **High** (60.000–100.000+)
- Batas keanggotaan dioptimalkan agar mencakup mobil berjarak tempuh ekstrim (hingga 200.000 km+) tanpa menghasilkan nilai 0.

In [11]:
# Visualisasi Himpunan Fuzzy Mileage
x = np.linspace(0, 250000, 1000)
low = [triangleMember(i, -40000, 0, 40000) for i in x]
medium = [triangleMember(i, 30000, 50000, 70000) for i in x]
high = [triangleMember(i, 60000, 200000, 400000) for i in x]

plt.plot(x, low, label="Low")
plt.plot(x, medium, label="Medium")
plt.plot(x, high, label="High")
plt.title("Mileage Membership (Optimized)")
plt.xlabel("Mileage")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [12]:
def fuzzify_mileage(mileage):
    low = triangleMember(mileage, -40000, 0, 40000)
    medium = triangleMember(mileage, 30000, 50000, 70000)
    high = triangleMember(mileage, 60000, 200000, 400000)
    return {"Low": low, "Medium": medium, "High": high}

fuzzify_mileage(35000)

#### 3.2. Variabel Input: Year (Tahun Mobil)
- Kategori: **Old** (2010–2018), **Medium** (2015–2021), **New** (2018–2023)
- Batas keanggotaan diperluas agar mencakup tahun mobil terlama dan terbaru (2024/2025).

In [13]:
# Visualisasi Himpunan Fuzzy Year
x = np.linspace(2005, 2026, 1000)
old = [triangleMember(i, 2000, 2010, 2018) for i in x]
medium = [triangleMember(i, 2015, 2018, 2021) for i in x]
new = [triangleMember(i, 2018, 2023, 2030) for i in x]

plt.plot(x, old, label="Old")
plt.plot(x, medium, label="Medium")
plt.plot(x, new, label="New")
plt.title("Year Membership (Optimized)")
plt.xlabel("Year")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [14]:
def fuzzify_year(year):
    old = triangleMember(year, 2000, 2010, 2018)
    medium = triangleMember(year, 2015, 2018, 2021)
    new = triangleMember(year, 2018, 2023, 2030)
    return {"Old": old, "Medium": medium, "New": new}

fuzzify_year(2019)

#### 3.3. Variabel Input: Engine Size (Kapasitas Mesin)
- Kategori: **Kecil** (0–2.0), **Sedang** (1.5–3.5), **Besar** (3.0–6.0+)

In [15]:
# Visualisasi Himpunan Fuzzy Engine Size
x = np.linspace(0, 7.0, 1000)
kecil = [triangleMember(i, -1.0, 1.0, 2.0) for i in x]
sedang = [triangleMember(i, 1.5, 2.5, 3.5) for i in x]
besar = [triangleMember(i, 3.0, 4.5, 8.0) for i in x]

plt.plot(x, kecil, label="Kecil")
plt.plot(x, sedang, label="Sedang")
plt.plot(x, besar, label="Besar")
plt.title("Engine Size Membership (Optimized)")
plt.xlabel("Engine Size")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [16]:
def fuzzify_engine(engine_size):
    kecil = triangleMember(engine_size, -1.0, 1.0, 2.0)
    sedang = triangleMember(engine_size, 1.5, 2.5, 3.5)
    besar = triangleMember(engine_size, 3.0, 4.5, 8.0)
    return {"Kecil": kecil, "Sedang": sedang, "Besar": besar}

fuzzify_engine(2.5)

#### 3.4. Variabel Input: MPG (Konsumsi Bahan Bakar)
- Kategori: **Boros** (0–40), **Standar** (30–70), **Irit** (60–80+)

In [17]:
# Visualisasi Himpunan Fuzzy MPG
x = np.linspace(0, 120, 1000)
boros = [triangleMember(i, -20, 20, 40) for i in x]
standar = [triangleMember(i, 30, 50, 70) for i in x]
irit = [triangleMember(i, 60, 70, 120) for i in x]

plt.plot(x, boros, label="Boros")
plt.plot(x, standar, label="Standar")
plt.plot(x, irit, label="Irit")
plt.title("MPG Membership (Optimized)")
plt.xlabel("MPG")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [18]:
def fuzzify_mpg(mpg):
    boros = triangleMember(mpg, -20, 20, 40)
    standar = triangleMember(mpg, 30, 50, 70)
    irit = triangleMember(mpg, 60, 70, 120)
    return {"Boros": boros, "Standar": standar, "Irit": irit}

fuzzify_mpg(55)

#### 3.5. Variabel Input: Tax (Pajak Tahunan)
- Kategori: **Murah** (0–200), **Standar** (150–450), **Mahal** (400–600+)

In [19]:
# Visualisasi Himpunan Fuzzy Tax
x = np.linspace(0, 1000, 1000)
murah = [triangleMember(i, -100, 100, 200) for i in x]
standar = [triangleMember(i, 150, 300, 450) for i in x]
mahal = [triangleMember(i, 400, 600, 1500) for i in x]

plt.plot(x, murah, label="Murah")
plt.plot(x, standar, label="Standar")
plt.plot(x, mahal, label="Mahal")
plt.title("Tax Membership (Optimized)")
plt.xlabel("Tax")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [20]:
def fuzzify_tax(tax):
    murah = triangleMember(tax, -100, 100, 200)
    standar = triangleMember(tax, 150, 300, 450)
    mahal = triangleMember(tax, 400, 600, 1500)
    return {"Murah": murah, "Standar": standar, "Mahal": mahal}

fuzzify_tax(250)

#### 3.6. Variabel Output: Price (Harga) — Ditambah Kategori Premium
Untuk menyelaraskan keluaran model fuzzy dengan evaluasi kelas harga riil, kita menambahkan himpunan output baru bernama **Premium**. Sekarang daerah fuzzy kontinu output Mamdani terdiri atas 4 kelas:
- **Murah** (0, 12.000, 22.000) $
ightarrow$ memetakan ke *Economy*
- **Standar** (18.000, 28.000, 38.000) $
ightarrow$ memetakan ke *Standard*
- **Premium** (32.000, 42.000, 52.000) $
ightarrow$ memetakan ke *Premium*
- **Mewah** (48.000, 75.000, 120.000) $
ightarrow$ memetakan ke *Luxury*

In [21]:
# Visualisasi Himpunan Fuzzy Price (Mamdani Output - 4 Kelas)
x = np.linspace(0, 120000, 1000)
murah = [triangleMember(i, 0, 3500, 15000) for i in x]
standar = [triangleMember(i, 15000, 23500, 30000) for i in x]
premium = [triangleMember(i, 26000, 28500, 40000) for i in x]
mewah = [triangleMember(i, 45000, 75000, 150000) for i in x]

plt.plot(x, murah, label="Murah")
plt.plot(x, standar, label="Standar")
plt.plot(x, premium, label="Premium")
plt.plot(x, mewah, label="Mewah")
plt.title("Price Membership (Mamdani Output - 4 Classes)")
plt.xlabel("Price ($)")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [22]:
def fuzzify_price(price):
    murah = triangleMember(price, 0, 3500, 15000)
    standar = triangleMember(price, 15000, 23500, 30000)
    premium = triangleMember(price, 26000, 28500, 40000)
    mewah = triangleMember(price, 45000, 75000, 150000)
    return {"Murah": murah, "Standar": standar, "Premium": premium, "Mewah": mewah}

fuzzify_price(28000)

### Langkah 4: Definisikan Aturan Fuzzy (Rule Base) & Inferensi
Kita memperbarui target output dari **20 aturan fuzzy** agar mencakup kelas output **Premium**. Penggabungan implikasi/agregasi Mamdani sekarang menghasilkan 4 himpunan fuzzy.

In [23]:
def rule_evaluation_all(year, mileage, engine_size, mpg, tax):
    year_fuzzy = fuzzify_year(year)
    mileage_fuzzy = fuzzify_mileage(mileage)
    engine_fuzzy = fuzzify_engine(engine_size)
    mpg_fuzzy = fuzzify_mpg(mpg)
    tax_fuzzy = fuzzify_tax(tax)

    # 20 Aturan Inferensi Fuzzy yang diselaraskan ke 4 Kelas
    rule1 = min(year_fuzzy["New"], mileage_fuzzy["Low"], engine_fuzzy["Besar"])
    rule2 = min(year_fuzzy["Old"], mileage_fuzzy["High"])
    rule3 = min(mpg_fuzzy["Irit"], tax_fuzzy["Murah"])
    rule4 = min(year_fuzzy["New"], mileage_fuzzy["Low"], mpg_fuzzy["Irit"])
    rule5 = min(engine_fuzzy["Besar"], tax_fuzzy["Mahal"])
    rule6 = min(year_fuzzy["Old"], mileage_fuzzy["High"], mpg_fuzzy["Boros"])
    rule7 = min(engine_fuzzy["Kecil"], mpg_fuzzy["Irit"])
    rule8 = min(tax_fuzzy["Mahal"], mileage_fuzzy["Low"])
    rule9 = min(year_fuzzy["Medium"], mileage_fuzzy["Medium"])
    rule10 = min(engine_fuzzy["Kecil"], mileage_fuzzy["High"])
    rule11 = min(year_fuzzy["New"], engine_fuzzy["Besar"])
    rule12 = min(year_fuzzy["Old"], tax_fuzzy["Murah"])
    rule13 = min(mpg_fuzzy["Boros"], tax_fuzzy["Mahal"])
    rule14 = min(mileage_fuzzy["Low"], mpg_fuzzy["Irit"])
    rule15 = min(engine_fuzzy["Sedang"], year_fuzzy["Medium"])
    rule16 = min(year_fuzzy["Medium"], mpg_fuzzy["Standar"])
    rule17 = min(mileage_fuzzy["Medium"], tax_fuzzy["Standar"])
    rule18 = min(engine_fuzzy["Sedang"], mileage_fuzzy["Medium"])
    rule19 = min(year_fuzzy["New"], engine_fuzzy["Sedang"])
    rule20 = min(year_fuzzy["Old"], engine_fuzzy["Kecil"])

    rules_list = [
        rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10,
        rule11, rule12, rule13, rule14, rule15, rule16, rule17, rule18, rule19, rule20
    ]

    # Agregasi implikasi Mamdani dengan 4 Output
    agg_dict = {
        "Murah": max(rule2, rule6, rule10, rule12, rule13, rule20),
        "Standar": max(rule3, rule7, rule9, rule15, rule16, rule17),
        "Premium": max(rule4, rule14, rule18, rule19),
        "Mewah": max(rule1, rule5, rule8, rule11)
    }

    return agg_dict, rules_list

def rule_evaluation(year, mileage, engine_size, mpg, tax):
    agg_dict, _ = rule_evaluation_all(year, mileage, engine_size, mpg, tax)
    return agg_dict

In [24]:
# Uji coba evaluasi aturan fuzzy
rule_evaluation(2020, 15000, 4.5, 60, 150)

### Langkah 5: Defuzzifikasi (Mamdani & Sugeno)
Defuzzifikasi mengubah output fuzzy menjadi nilai tegas (crisp price). Kita mengimplementasikan dua metode:

1. **Fuzzy Mamdani (Centroid / COA)**:
   Mencari titik pusat gravitasi area luaran fuzzy kontinu secara numerik di sepanjang semesta \$0 s.d. \$100.000 dengan langkah \$500.
   
2. **Fuzzy Sugeno (Weighted Average)**:
   Menggunakan nilai output tegas konstan (*singleton/konstanta*) yang dioptimalkan sesuai kelas target:
   - Output Murah = \$12.000
   - Output Standar = \$28.000
   - Output Premium = \$42.000
   - Output Mewah = \$75.000

In [25]:
def defuzzify_mamdani(output_fuzzy):
    w_murah = output_fuzzy["Murah"]
    w_standar = output_fuzzy["Standar"]
    w_premium = output_fuzzy["Premium"]
    w_mewah = output_fuzzy["Mewah"]
    
    # Diskritisasi harga dari $0 s.d. $100,000 dengan langkah 500
    y_vals = np.arange(0, 100001, 500)
    numerator = 0
    denominator = 0
    
    for y in y_vals:
        mu_murah = triangleMember(y, 0, 3500, 15000)
        mu_standar = triangleMember(y, 15000, 23500, 30000)
        mu_premium = triangleMember(y, 26000, 28500, 40000)
        mu_mewah = triangleMember(y, 45000, 75000, 150000)
        
        # Clipping (min) dan Aggregasi (max)
        mu_y = max(min(w_murah, mu_murah), min(w_standar, mu_standar), min(w_premium, mu_premium), min(w_mewah, mu_mewah))
        
        numerator += y * mu_y
        denominator += mu_y
        
    if denominator == 0:
        return 0
    return numerator / denominator

def defuzzify_sugeno(rules):
    # Singleton yang dioptimalkan untuk masing-masing 20 aturan
    z = [
        75000, 3500, 23500, 28500, 75000, 3500, 23500, 75000, 23500, 3500,
        75000, 3500, 3500, 28500, 23500, 23500, 23500, 28500, 28500, 3500
    ]
    numerator = sum(rules[i] * z[i] for i in range(20))
    denominator = sum(rules)
    if denominator == 0:
        return 0
    return numerator / denominator

def defuzzification(output_fuzzy):
    return defuzzify_mamdani(output_fuzzy)

In [26]:
# Menghitung estimasi harga tegas
hasil_fuzzy = rule_evaluation(2020, 15000, 4.5, 60, 150)
prediksi_mamdani = defuzzification(hasil_fuzzy)
print("Estimasi Harga Mamdani Centroid:", prediksi_mamdani)

Estimasi Harga Mamdani Centroid: 66290.69576321561


### Langkah 6: Prediksi Harga & Pengkategorian Mobil
Membuat fungsi pembungkus (wrapper) prediksi untuk memprediksi harga berdasarkan input numerik riil dan metode pilihan. Kita juga mengkategorikan mobil bekas menjadi:
- **Economy** (harga < \$20.000)
- **Standard** (harga \$20.000 s.d. \$34.999)
- **Premium** (harga \$35.000 s.d. \$49.999)
- **Luxury** (harga >= \$50.000)

In [27]:
def prediksi_harga(year, mileage, engine_size, mpg, tax, method="Mamdani"):
    agg_dict, rules_list = rule_evaluation_all(year, mileage, engine_size, mpg, tax)
    if method.lower() == "sugeno":
        return defuzzify_sugeno(rules_list)
    else:
        return defuzzify_mamdani(agg_dict)

def kategori_mobil(harga):
    if harga < 20000:
        return "Economy"
    elif harga < 35000:
        return "Standard"
    elif harga < 50000:
        return "Premium"
    else:
        return "Luxury"

In [28]:
# Menghitung prediksi final untuk Mercedes C-Class
harga = prediksi_harga(2020, 15000, 4.5, 60, 150, method="Mamdani")
kategori = kategori_mobil(harga)

print("=======- HASIL PREDIKSI KENDARAAN -=======")
print("Estimasi Harga Mobil : $", round(harga, 2))
print("Kategori Mobil       :", kategori)

=======- HASIL PREDIKSI KENDARAAN -=======
Estimasi Harga Mobil : $ 66290.7
Kategori Mobil       : Luxury


### Langkah 7: Pengujian & Evaluasi Performa (Mamdani vs Sugeno)
Kita membandingkan hasil prediksi kedua model fuzzy *from scratch* ini dengan harga riil pada **seluruh Test Set** (data uji yang belum pernah digunakan) untuk menghitung metrik error secara komprehensif **Mean Absolute Error (MAE)**, **Mean Squared Error (MSE)**, dan **Root Mean Squared Error (RMSE)**. Terakhir, kita mengevaluasi akurasi prediksi kategori menggunakan **Confusion Matrix**.

In [29]:
harga_asli = []
harga_prediksi_mamdani = []
harga_prediksi_sugeno = []
sample_data = test_data  # Evaluasi pada SELURUH test set


In [30]:
total = len(sample_data)
for idx, (index, row) in enumerate(sample_data.iterrows()):
    pred_mamdani = prediksi_harga(row["year"], row["mileage"], row["engineSize"], row["mpg"], row["tax"], method="Mamdani")
    pred_sugeno = prediksi_harga(row["year"], row["mileage"], row["engineSize"], row["mpg"], row["tax"], method="Sugeno")
    
    harga_prediksi_mamdani.append(pred_mamdani)
    harga_prediksi_sugeno.append(pred_sugeno)
    harga_asli.append(row["price"])
    
    if (idx + 1) % 500 == 0 or (idx + 1) == total:
        print(f"Progress: {idx + 1}/{total} data selesai dievaluasi...")

print(f"\nTotal data yang dievaluasi: {total}")


Progress: 500/2572 data selesai dievaluasi...
Progress: 1000/2572 data selesai dievaluasi...
Progress: 1500/2572 data selesai dievaluasi...
Progress: 2000/2572 data selesai dievaluasi...
Progress: 2500/2572 data selesai dievaluasi...
Progress: 2572/2572 data selesai dievaluasi...

Total data yang dievaluasi: 2572


In [31]:
print("Perbandingan Prediksi Harga Mobil Bekas (Mamdani vs Sugeno):")
print(f"Menampilkan 10 sampel pertama dari {len(harga_asli)} data uji:\n")

for i in range(min(10, len(harga_asli))):
    kat_asli = kategori_mobil(harga_asli[i])
    kat_mamdani = kategori_mobil(harga_prediksi_mamdani[i])
    kat_sugeno = kategori_mobil(harga_prediksi_sugeno[i])
    print(f"Data ke-{i+1:02d} | Asli: ${harga_asli[i]:>10,.2f} ({kat_asli})")
    print(f"           | Mamdani: ${harga_prediksi_mamdani[i]:>10,.2f} ({kat_mamdani})")
    print(f"           | Sugeno:  ${harga_prediksi_sugeno[i]:>10,.2f} ({kat_sugeno})")
    print("-" * 55)

print(f"\n... dan {len(harga_asli) - 10} data lainnya (total {len(harga_asli)} data)")


Perbandingan Prediksi Harga Mobil Bekas (Mamdani vs Sugeno):
Menampilkan 10 sampel pertama dari 2572 data uji:

Data ke-01 | Asli: $ 23,999.00 (Standard)
           | Mamdani: $ 25,292.03 (Standard)
           | Sugeno:  $ 24,231.71 (Standard)
-------------------------------------------------------
Data ke-02 | Asli: $ 16,990.00 (Economy)
           | Mamdani: $ 27,879.42 (Standard)
           | Sugeno:  $ 26,500.00 (Standard)
-------------------------------------------------------
Data ke-03 | Asli: $ 23,499.00 (Standard)
           | Mamdani: $ 25,292.03 (Standard)
           | Sugeno:  $ 24,231.71 (Standard)
-------------------------------------------------------
Data ke-04 | Asli: $ 14,868.00 (Economy)
           | Mamdani: $ 21,621.68 (Standard)
           | Sugeno:  $ 19,535.29 (Economy)
-------------------------------------------------------
Data ke-05 | Asli: $ 25,830.00 (Standard)
           | Mamdani: $ 24,477.72 (Standard)
           | Sugeno:  $ 24,316.33 (Standard)
-------

#### 7.0. Implementasi Metrik Evaluasi From Scratch
Sebelum menggunakan pustaka `sklearn` sebagai validasi, kita menghitung metrik MAE, MSE, dan RMSE secara manual untuk membuktikan pemahaman rumus.


In [32]:
# === IMPLEMENTASI METRIK FROM SCRATCH ===

# MAE (Mean Absolute Error) = (1/n) * Σ|actual - predicted|
def hitung_mae(actual, predicted):
    n = len(actual)
    total_error = 0
    for i in range(n):
        total_error += abs(actual[i] - predicted[i])
    return total_error / n

# MSE (Mean Squared Error) = (1/n) * Σ(actual - predicted)²
def hitung_mse(actual, predicted):
    n = len(actual)
    total_error = 0
    for i in range(n):
        total_error += (actual[i] - predicted[i]) ** 2
    return total_error / n

# RMSE (Root Mean Squared Error) = √MSE
def hitung_rmse(actual, predicted):
    return hitung_mse(actual, predicted) ** 0.5

# Hitung metrik manual
mae_mamdani_manual = hitung_mae(harga_asli, harga_prediksi_mamdani)
mae_sugeno_manual = hitung_mae(harga_asli, harga_prediksi_sugeno)
mse_mamdani_manual = hitung_mse(harga_asli, harga_prediksi_mamdani)
mse_sugeno_manual = hitung_mse(harga_asli, harga_prediksi_sugeno)
rmse_mamdani_manual = hitung_rmse(harga_asli, harga_prediksi_mamdani)
rmse_sugeno_manual = hitung_rmse(harga_asli, harga_prediksi_sugeno)

print("=== EVALUASI METRIK (IMPLEMENTASI FROM SCRATCH) ===")
print(f"MAE  Mamdani : {mae_mamdani_manual:>12,.2f}    |  MAE  Sugeno : {mae_sugeno_manual:>12,.2f}")
print(f"MSE  Mamdani : {mse_mamdani_manual:>16,.2f}  |  MSE  Sugeno : {mse_sugeno_manual:>16,.2f}")
print(f"RMSE Mamdani : {rmse_mamdani_manual:>12,.2f}    |  RMSE Sugeno : {rmse_sugeno_manual:>12,.2f}")


=== EVALUASI METRIK (IMPLEMENTASI FROM SCRATCH) ===
MAE  Mamdani :     6,817.37    |  MAE  Sugeno :     6,574.06
MSE  Mamdani :   100,336,508.85  |  MSE  Sugeno :    96,956,988.47
RMSE Mamdani :    10,016.81    |  RMSE Sugeno :     9,846.67


In [33]:
mae_mamdani = mean_absolute_error(harga_asli, harga_prediksi_mamdani)
mae_sugeno = mean_absolute_error(harga_asli, harga_prediksi_sugeno)
print(f"MAE Mamdani : {mae_mamdani:,.2f}")
print(f"MAE Sugeno  : {mae_sugeno:,.2f}")

MAE Mamdani : 6,817.37
MAE Sugeno  : 6,574.06


In [34]:
mse_mamdani = mean_squared_error(harga_asli, harga_prediksi_mamdani)
mse_sugeno = mean_squared_error(harga_asli, harga_prediksi_sugeno)
print(f"MSE Mamdani : {mse_mamdani:,.2f}")
print(f"MSE Sugeno  : {mse_sugeno:,.2f}")

MSE Mamdani : 100,336,508.85
MSE Sugeno  : 96,956,988.47


In [35]:
rmse_mamdani = np.sqrt(mse_mamdani)
rmse_sugeno = np.sqrt(mse_sugeno)
print(f"RMSE Mamdani: {rmse_mamdani:,.2f}")
print(f"RMSE Sugeno : {rmse_sugeno:,.2f}")

RMSE Mamdani: 10,016.81
RMSE Sugeno : 9,846.67


#### 7.1. Confusion Matrix Kategori Harga
Untuk mengevaluasi seberapa akurat sistem memetakan mobil bekas ke dalam kategori kelas yang sesuai, kita mengubah nilai nominal (harga asli, prediksi Mamdani, dan prediksi Sugeno) menjadi label kategori (*Economy*, *Standard*, *Premium*, *Luxury*), lalu merancang **Confusion Matrix**.

In [36]:
# Mengonversi nilai real menjadi kategori string
kategori_asli = [kategori_mobil(h) for h in harga_asli]
kategori_pred_mamdani = [kategori_mobil(h) for h in harga_prediksi_mamdani]
kategori_pred_sugeno = [kategori_mobil(h) for h in harga_prediksi_sugeno]

# List label kategori agar urutan sumbu pada matriks konsisten
labels = ["Economy", "Standard", "Premium", "Luxury"]

# Hitung confusion matrix
cm_mamdani = confusion_matrix(kategori_asli, kategori_pred_mamdani, labels=labels)
cm_sugeno = confusion_matrix(kategori_asli, kategori_pred_sugeno, labels=labels)

# Visualisasi side-by-side menggunakan Matplotlib
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot Mamdani
disp_mamdani = ConfusionMatrixDisplay(confusion_matrix=cm_mamdani, display_labels=labels)
disp_mamdani.plot(cmap="Blues", ax=ax[0])
ax[0].set_title("Confusion Matrix - Fuzzy Mamdani")

# Plot Sugeno
disp_sugeno = ConfusionMatrixDisplay(confusion_matrix=cm_sugeno, display_labels=labels)
disp_sugeno.plot(cmap="Greens", ax=ax[1])
ax[1].set_title("Confusion Matrix - Fuzzy Sugeno")

plt.tight_layout()
plt.show()

# Hitung dan Tampilkan Akurasi Klasifikasi Kategori
acc_mamdani = accuracy_score(kategori_asli, kategori_pred_mamdani) * 100
acc_sugeno = accuracy_score(kategori_asli, kategori_pred_sugeno) * 100
print(f"Akurasi Klasifikasi Kategori Mamdani: {acc_mamdani:.2f}%")
print(f"Akurasi Klasifikasi Kategori Sugeno : {acc_sugeno:.2f}%")


Akurasi Klasifikasi Kategori Mamdani: 57.85%
Akurasi Klasifikasi Kategori Sugeno : 63.30%


#### 7.2. Visualisasi Scatter Plot — Harga Asli vs Prediksi
Scatter plot menunjukkan seberapa dekat prediksi fuzzy dengan harga asli. Garis merah putus-putus menunjukkan prediksi ideal (prediksi = asli). Semakin dekat titik-titik ke garis, semakin akurat model.


In [37]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Mamdani
axes[0].scatter(harga_asli, harga_prediksi_mamdani, alpha=0.3, s=10, color='#1f77b4')
axes[0].plot([0, max(harga_asli)], [0, max(harga_asli)], 'r--', linewidth=1, label='Prediksi Ideal')
axes[0].set_xlabel('Harga Asli ($)')
axes[0].set_ylabel('Harga Prediksi Mamdani ($)')
axes[0].set_title('Scatter Plot — Fuzzy Mamdani')
axes[0].legend()
axes[0].set_xlim(0, max(harga_asli) * 1.05)
axes[0].set_ylim(0, max(harga_prediksi_mamdani) * 1.05)

# Plot Sugeno
axes[1].scatter(harga_asli, harga_prediksi_sugeno, alpha=0.3, s=10, color='#2ca02c')
axes[1].plot([0, max(harga_asli)], [0, max(harga_asli)], 'r--', linewidth=1, label='Prediksi Ideal')
axes[1].set_xlabel('Harga Asli ($)')
axes[1].set_ylabel('Harga Prediksi Sugeno ($)')
axes[1].set_title('Scatter Plot — Fuzzy Sugeno')
axes[1].legend()
axes[1].set_xlim(0, max(harga_asli) * 1.05)
axes[1].set_ylim(0, max(harga_prediksi_sugeno) * 1.05)

plt.tight_layout()
plt.show()


# --- Analisis Perbandingan Mamdani vs Sugeno

Berdasarkan hasil pengujian di atas, berikut adalah perbandingan antara Metode Fuzzy Mamdani dan Sugeno:

### 1. Perbedaan Hasil Output & Performa (Akurasi/Error)
- **Hasil Output**: Hasil prediksi dari Fuzzy Mamdani dan Fuzzy Sugeno menunjukkan sedikit perbedaan nominal harga. Hal ini disebabkan oleh perbedaan mendasar pada proses defuzzifikasi.
- **Akurasi Klasifikasi Kategori**: Dari seluruh data uji Test Set (2.572 sampel), metode Fuzzy Mamdani menghasilkan **Akurasi Kategori sebesar 57.85%** (1.488 dari 2.572 sampel benar), sedangkan Fuzzy Sugeno menghasilkan **Akurasi Kategori sebesar 63.30%** (1.628 dari 2.572 sampel benar) dalam memetakan harga prediksi ke kategori kelas harga mobil (Economy, Standard, Premium, Luxury). Evaluasi pada seluruh test set memberikan gambaran performa yang lebih representatif.
- **Evaluasi Metrik (MAE & MSE)**: Metode yang menghasilkan error lebih rendah bervariasi bergantung pada kecocokan singleton/konstanta output pada Sugeno terhadap sebaran data riil. Sugeno cenderung lebih konsisten karena sifatnya yang linier/konstan tanpa dipengaruhi oleh bentuk kurva keanggonaan output.

### 2. Interpretasi Kelebihan dan Kekurangan masing-masing metode

| Aspek Perbandingan | Fuzzy Mamdani | Fuzzy Sugeno |
| :--- | :--- | :--- |
| **Format Output** | Berupa himpunan fuzzy (kurva kontinu/segitiga/trapesium). | Berupa nilai konstanta (singleton) atau persamaan linier. |
| **Proses Defuzzifikasi** | Menggunakan metode **Centroid** (Center of Area) yang membutuhkan diskritisasi/integrasi. | Menggunakan rata-rata terbobot (**Weighted Average**) yang langsung menghitung nilai tegas. |
| **Akurasi Kategori (Full Test Set)** | **57.85%** (1.488 dari 2.572 benar) | **63.30%** (1.628 dari 2.572 benar) |
| **Beban Komputasi** | **Lebih Tinggi**: Membutuhkan iterasi/diskritisasi untuk mencari titik pusat gravitasi pada area output. | **Sangat Rendah**: Rumus matematika sederhana langsung menghasilkan nilai tegas dengan cepat. |
| **Kecocokan Input/Output** | Sangat cocok untuk menangani persepsi kualitatif manusia yang bersifat subjektif. | Sangat cocok untuk optimasi matematis, kontroler industri, dan integrasi dengan Machine Learning. |
| **Intuitif bagi Manusia** | Lebih intuitif dan mudah dipahami karena seluruh variabel (termasuk output) memiliki grafik linguistik. | Kurang intuitif untuk outputnya karena langsung diwakili oleh angka konstanta/singleton. |
